In [1]:
# Example for running a PyTorch repo in Kaggle
!pip install -r requirements.txt
# Or install specific missing dependencies:
!pip install timm==0.4.12 opencv-python scikit-learn

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.0/377.0 kB 6.9 MB/s eta 0:00:00
  Attempting uninstall: timm
    Found existing installation: timm 1.0.26
    Uninstalling timm-1.0.26:
      Successfully uninstalled timm-1.0.26


In [2]:
# ============================================================
# CELL 1: SETUP & ENVIRONMENT
# ============================================================
import os
import subprocess
import sys
from pathlib import Path

# --- Reproducibility ---
SEED = 42

# --- Directory structure ---
WORKING_DIR = Path("/kaggle/working")
RESULTS_DIR = WORKING_DIR / "results"
REPO_DIR = WORKING_DIR / "PraNet"
WEIGHTS_DIR = WORKING_DIR / "pranetweight"

for d in (RESULTS_DIR, WEIGHTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

def run(cmd: str) -> None:
    """Run a shell command and raise on failure."""
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed ({result.returncode}): {cmd}")

# --- Clone PraNet repo ---
if not REPO_DIR.exists():
    run(f"git clone https://github.com/DengPingFan/PraNet.git {REPO_DIR}")
else:
    print(f"Repo already present at {REPO_DIR}")

# Add repo to sys.path so 'lib' imports succeed
sys.path.insert(0, str(REPO_DIR))

# --- Install dependencies ---
run("pip install -q opencv-python-headless scikit-image imageio thop")

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T

torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE} | Repo: {REPO_DIR} | Results: {RESULTS_DIR}")

$ git clone https://github.com/DengPingFan/PraNet.git /kaggle/working/PraNet


Cloning into '/kaggle/working/PraNet'...


$ pip install -q opencv-python-headless scikit-image imageio thop
Device: cpu | Repo: /kaggle/working/PraNet | Results: /kaggle/working/results


In [3]:
# ============================================================
# CELL 2: INFERENCE DATASET PREPARATION
# ============================================================
import glob
import os
from pathlib import Path

IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

DATASET_KEYWORDS = {
    "Kvasir": ["kvasir"],
    "CVC-ClinicDB": ["cvc-clinicdb"],
    "CVC-300": ["cvc-300"],
    "CVC-ColonDB": ["cvc-colondb"],
    "ETIS-LaribPolypDB": ["etis-laribpolypdb"],
}

def find_dataset_root(keywords: list[str], search_root: str = "/kaggle/input") -> str | None:
    for dirpath, _, _ in os.walk(search_root):
        lower = dirpath.lower()
        if any(kw in lower for kw in keywords):
            return dirpath
    return None

def find_images(root: str) -> list[str]:
    """Find all image paths inside dataset root."""
    images = []
    for dirpath, _, filenames in os.walk(root):
        for f in filenames:
            if f.lower().endswith(IMG_EXTS):
                images.append(os.path.join(dirpath, f))
    return sorted(images)

def build_dataset_registry() -> dict:
    registry = {}
    for name, keywords in DATASET_KEYWORDS.items():
        root = find_dataset_root(keywords)
        if root is None:
            print(f"[WARN] Dataset '{name}' not found under /kaggle/input — skipping.")
            continue
        images = find_images(root)
        if not images:
            print(f"[WARN] '{name}' root found at {root} but no images matched.")
            continue
        registry[name] = {"root": root, "images": images}
        print(f"[OK] {name}: {len(images)} images found at {root}")
    return registry

DATASET_REGISTRY = build_dataset_registry()

[OK] Kvasir: 100 images found at /kaggle/input/datasets/roopasanjay/offmaps/PraNet/Kvasir
[OK] CVC-ClinicDB: 62 images found at /kaggle/input/datasets/roopasanjay/offmaps/PraNet/CVC-ClinicDB
[OK] CVC-300: 60 images found at /kaggle/input/datasets/roopasanjay/offmaps/PraNet/CVC-300
[OK] CVC-ColonDB: 380 images found at /kaggle/input/datasets/roopasanjay/offmaps/PraNet/CVC-ColonDB
[OK] ETIS-LaribPolypDB: 196 images found at /kaggle/input/datasets/roopasanjay/offmaps/PraNet/ETIS-LaribPolypDB


In [4]:
# ============================================================
# CELL 3: MODEL ARCHITECTURE & WEIGHTS LOADER
# ============================================================
import os
import torch

# 1. Import PraNet and its modules
import lib.Res2Net_v1b as res2net_module
import lib.PraNet_Res2Net as pranet_module
from lib.PraNet_Res2Net import PraNet

# 2. Patch the function inside BOTH modules directly
no_pretrained_res2net = lambda pretrained=False, **kwargs: res2net_module.Res2Net(
    res2net_module.Bottle2neck, [3, 4, 6, 3], baseWidth=26, scale=4, **kwargs
)
res2net_module.res2net50_v1b_26w_4s = no_pretrained_res2net
pranet_module.res2net50_v1b_26w_4s = no_pretrained_res2net


def find_weights_file(explicit_path: str | None = None) -> str:
    """Locate PraNet .pth weights."""
    if explicit_path and os.path.exists(explicit_path):
        return explicit_path
    search_dirs = ["/kaggle/input", str(WEIGHTS_DIR)]
    for base in search_dirs:
        for dirpath, _, filenames in os.walk(base):
            for f in filenames:
                if f.lower().endswith(".pth") and "pranet" in f.lower():
                    return os.path.join(dirpath, f)
    raise FileNotFoundError("No PraNet weights (.pth) found.")


def load_pranet(weights_path: str | None = None, device: torch.device = DEVICE) -> torch.nn.Module:
    """Instantiate PraNet, load pretrained weights safely, and set eval mode."""
    resolved_path = find_weights_file(weights_path)
    print(f"Loading PraNet weights from: {resolved_path}")
    
    model = PraNet()
    state_dict = torch.load(resolved_path, map_location=device)
    
    if "state_dict" in state_dict:
        state_dict = state_dict["state_dict"]
    elif "model" in state_dict:
        state_dict = state_dict["model"]
        
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    return model


# Instantiate the model so Cell 5 can use it
model = load_pranet()

Loading PraNet weights from: /kaggle/input/datasets/roopasanjay/pranetweigh/PraNet-19 (1).pth


In [5]:
# ============================================================
# CELL 4: INFERENCE & PREDICTION PIPELINE
# ============================================================
import cv2
import torch
import torchvision.transforms as T
from torchvision.transforms import InterpolationMode

IMG_SIZE = 352
img_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE), interpolation=InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class PolypInferenceDataset(Dataset):
    def __init__(self, image_paths: list[str]):
        self.image_paths = image_paths

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, idx: int):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        orig_size = image.size  # (Width, Height)
        image_t = img_transform(image)
        return image_t, img_path, orig_size

@torch.no_grad()
def run_inference(
    model: torch.nn.Module,
    image_paths: list[str],
    output_dir: Path,
    device: torch.device = DEVICE,
    batch_size: int = 8,
    threshold: float = 0.5,
):
    output_dir.mkdir(parents=True, exist_ok=True)
    dataset = PolypInferenceDataset(image_paths)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    for images, paths, (widths, heights) in loader:
        images = images.to(device)
        preds = model(images)
        
        # PraNet outputs multiple lateral maps; take the primary prediction (lateral_map-5)
        if isinstance(preds, (list, tuple)):
            preds = preds[0]
            
        probs = torch.sigmoid(preds)

        for i in range(len(paths)):
            prob_map = probs[i, 0].cpu().numpy()
            orig_w, orig_h = widths[i].item(), heights[i].item()

            # Resize probability map back to original dimensions
            prob_map_resized = cv2.resize(prob_map, (orig_w, orig_h), interpolation=cv2.INTER_LINEAR)
            
            # Normalize and threshold prediction
            norm_map = (prob_map_resized - prob_map_resized.min()) / (prob_map_resized.max() - prob_map_resized.min() + 1e-8)
            binary_mask = ((norm_map > threshold) * 255).astype(np.uint8)

            # Save binary mask
            save_path = output_dir / f"{Path(paths[i]).stem}_mask.png"
            cv2.imwrite(str(save_path), binary_mask)

In [6]:
# ============================================================
# CELL 5: EXECUTION
# ============================================================
BATCH_SIZE = 8
THRESHOLD = 0.5

for dataset_name, info in DATASET_REGISTRY.items():
    print(f"\n============================================================")
    print(f"Generating predictions for: {dataset_name} ({len(info['images'])} images)")
    print(f"============================================================")
    
    save_dir = RESULTS_DIR / dataset_name.lower()
    run_inference(
        model=model,
        image_paths=info["images"],
        output_dir=save_dir,
        batch_size=BATCH_SIZE,
        threshold=THRESHOLD,
    )
    print(f"[DONE] Saved predicted masks to: {save_dir}")


Generating predictions for: Kvasir (100 images)
[DONE] Saved predicted masks to: /kaggle/working/results/kvasir

Generating predictions for: CVC-ClinicDB (62 images)
[DONE] Saved predicted masks to: /kaggle/working/results/cvc-clinicdb

Generating predictions for: CVC-300 (60 images)
[DONE] Saved predicted masks to: /kaggle/working/results/cvc-300

Generating predictions for: CVC-ColonDB (380 images)
[DONE] Saved predicted masks to: /kaggle/working/results/cvc-colondb

Generating predictions for: ETIS-LaribPolypDB (196 images)
[DONE] Saved predicted masks to: /kaggle/working/results/etis-laribpolypdb
